In [ ]:
# !pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community

In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings


from langchain_chroma import Chroma

/home/foolmann/miniconda3/envs/genaienv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [4]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [5]:
vector_store = Chroma(
    embedding_function=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001"),
    persist_directory='my_chroma_db',
    collection_name='sample'
)

In [6]:
# add documents
vector_store.add_documents(docs)

['468a5c21-cc2d-448c-add1-b077557f4f46',
 'dd4d4971-2fc1-4f1b-953d-e6e344ed5401',
 'ca267aeb-b6f5-4342-b9d8-624b5da31608',
 '5b05aea4-5b53-4e87-90c2-032c824e4395',
 'c0ce0163-dd02-47bf-ae9d-9fb65b0bc14c']

In [7]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['468a5c21-cc2d-448c-add1-b077557f4f46',
  'dd4d4971-2fc1-4f1b-953d-e6e344ed5401',
  'ca267aeb-b6f5-4342-b9d8-624b5da31608',
  '5b05aea4-5b53-4e87-90c2-032c824e4395',
  'c0ce0163-dd02-47bf-ae9d-9fb65b0bc14c'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

In [8]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2  # no of the similar objects/documents we want. 
)

[Document(id='5b05aea4-5b53-4e87-90c2-032c824e4395', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='c0ce0163-dd02-47bf-ae9d-9fb65b0bc14c', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [9]:
# search with similarity score
vector_store.similarity_search_with_score( # smaller the distance score better
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='5b05aea4-5b53-4e87-90c2-032c824e4395', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6406819820404053),
 (Document(id='c0ce0163-dd02-47bf-ae9d-9fb65b0bc14c', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.660536527633667)]

In [12]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="Best Player",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='ca267aeb-b6f5-4342-b9d8-624b5da31608', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.7072809338569641),
 (Document(id='c0ce0163-dd02-47bf-ae9d-9fb65b0bc14c', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.724321722984314)]

In [16]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='468a5c21-cc2d-448c-add1-b077557f4f46', document=updated_doc1)


In [17]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['468a5c21-cc2d-448c-add1-b077557f4f46',
  'dd4d4971-2fc1-4f1b-953d-e6e344ed5401',
  'ca267aeb-b6f5-4342-b9d8-624b5da31608',
  '5b05aea4-5b53-4e87-90c2-032c824e4395',
  'c0ce0163-dd02-47bf-ae9d-9fb65b0bc14c'],
 'embeddings': array([[-0.00719311,  0.02029932,  0.02837158, ...,  0.01391   ,
         -0.01240376, -0.00345245],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]], shape=(5, 3072)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple c

In [18]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [19]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['468a5c21-cc2d-448c-add1-b077557f4f46',
  'dd4d4971-2fc1-4f1b-953d-e6e344ed5401',
  'ca267aeb-b6f5-4342-b9d8-624b5da31608',
  '5b05aea4-5b53-4e87-90c2-032c824e4395',
  'c0ce0163-dd02-47bf-ae9d-9fb65b0bc14c'],
 'embeddings': array([[-0.00719311,  0.02029932,  0.02837158, ...,  0.01391   ,
         -0.01240376, -0.00345245],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]], shape=(5, 3072)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple c